# Address Matching: загрузка данных и проверка в Colab
Запускайте ячейки сверху вниз. Начальная ячейка найдёт папку проекта или скачает репозиторий.
После сброса сессии повторите запуск с начала. Для OSM необходим интернет.
Этот ноутбук должен использовать обновлённый проект со скриптом `scripts/download_reference_osm.py`.


In [ ]:
from pathlib import Path
import subprocess
import sys

base = Path('/content') if Path('/content').is_dir() else Path.cwd()
# Если проект находится в другом месте, укажите путь здесь.
PROJECT_DIR = None

candidates = [Path(PROJECT_DIR)] if PROJECT_DIR else [
    Path.cwd(), *Path.cwd().parents,
    base / 'Address_matching_GPB_1',
    base / 'Address_matching_GPB_1-main',
    base / 'Address_matching_GPB',
    base / 'Address_matching_GPB-main',
]
project_dir = next((p.resolve() for p in candidates
                    if (p / 'src/hybrid.py').is_file()
                    and (p / 'requirements.txt').is_file()), None)
if project_dir is None:
    destination = base / 'Address_matching_GPB_1'
    if destination.exists():
        raise RuntimeError(f'{destination} существует, но нужные файлы не найдены. '
                           'Укажите PROJECT_DIR для правильной папки.')
    subprocess.run(['git', 'clone', 'https://github.com/AlexSDem/Address_matching_GPB_1.git',
                    str(destination)], check=True)
    project_dir = destination.resolve()
if not (project_dir / 'scripts/download_reference_osm.py').is_file():
    raise RuntimeError('В этой копии нет загрузчика OSM. Обновите проект файлами из нового архива '
                       'или загрузите актуальную версию в GitHub и выполните git pull.')
print('Папка проекта:', project_dir)
sys.path.insert(0, str(project_dir))


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '-r', str(project_dir / 'requirements-dev.txt'),
                '-r', str(project_dir / 'requirements-osm.txt')], check=True)


## 1. Проверка кода
Тесты используют локальные примеры и имитацию ответов OSM; они не скачивают реальные адреса.


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=project_dir, check=True)


## 2. Создание справочника
По умолчанию: Северное Тушино, Южное Тушино, Центральный район Санкт-Петербурга.
Список меняется в `data/osm_places.json`. Загрузка может занять несколько минут.
Существующий CSV используется повторно. Для обновления добавьте `--refresh` в список аргументов.
Если один из районов не загрузился, ячейка остановится: основной CSV не будет заменён неполными данными.
Подробности — в `reference_osm.attempt.json`. Повторите позже или явно добавьте `--allow-partial`.


In [ ]:
subprocess.run([sys.executable, str(project_dir / 'scripts/download_reference_osm.py')],
               cwd=project_dir, check=True)
reference_path = project_dir / 'reference_osm.csv'
print(reference_path)


## 3. Загрузка модели
Словарь строится по справочнику. Это подготовка поискового индекса, а не обучение нейросети.


In [ ]:
from src.io import read_reference, read_aliases
from src.hybrid import HybridAddressMatcher

reference = read_reference(reference_path)
print('Адресов:', len(reference))
display(reference.head())
matcher = HybridAddressMatcher().fit(reference, read_aliases())


In [ ]:
query = reference.iloc[0]['address']  # затем замените на свой адрес
result = matcher.match_one(query)
print('Запрос:', query)
print('Статус:', result.status)
print('Принятый адрес:', result.best)
print('Кандидат:', result.candidate)
print('Причины:', result.reasons)


## 4. Сохранение данных на компьютер
CSV и отчёт содержат результаты новой выгрузки, не исходный снимок из 6 448 адресов.
Реальный набор не является размеченным тестом: для оценки качества нужны отдельные запросы с правильными ID.
Данные: © OpenStreetMap contributors, https://www.openstreetmap.org/copyright


In [ ]:
try:
    from google.colab import files
except ImportError:
    print('Вне Colab файлы доступны здесь:', reference_path)
else:
    files.download(str(reference_path))
    metadata_path = reference_path.with_suffix('.meta.json')
    if metadata_path.exists():
        files.download(str(metadata_path))
